# NB6 · The web interface

**Building Clinical Decision Support Systems with Generative AI Tools**  
Technology and Artificial Intelligence Literacy Training in Health Sciences · Akdeniz University · 18 September 2026

Prof. Dr. Utku Köse · Süleyman Demirel University, Department of Computer Engineering  
Director, Artificial Intelligence Application and Research Center (YAZEM) · utkukose@sdu.edu.tr

---


## What this notebook does

The program you wrote across five notebooks works, but it can only be used from inside
Colab. In this notebook you will add a web interface to it. The interface produces a link
that opens in a browser, which makes it something you can show a clinician.

You will not display a single number. The decision, the state of the guardrails and the
reasoning appear together. This was the distinction drawn in the lecture on 16 September:
the product of a decision support system is not a diagnosis but a direction of attention,
presented with its reasoning.


## Setup


In [ ]:
!pip -q install pandas numpy scikit-learn matplotlib

import urllib.request

REPO = 'https://raw.githubusercontent.com/utkukose/cdss-genai-NB-lecture/main'
urllib.request.urlretrieve(f'{REPO}/workshop/cdss_kit.py', 'cdss_kit.py')

import cdss_kit as kit
kit.LANG = 'en'

print('Ready.')

!pip -q install gradio


---

## Code carried from the previous notebook

Paste the whole block collected at the end of the previous notebook into the cell
below. Do not delete the `#@cdss` marker on the first line; that block is collected
again at the end of this notebook and carried to the next one.

Running the block rebuilds everything you wrote in the earlier notebooks. Where it
reads data from the web, the cell may take a few seconds.


In [ ]:
#@cdss onceki_defter
# Paste the generated code below this line.


### Check · The carried code


In [ ]:
kit.check_defined('model', 'guarded_prediction', 'patient_reasoning',
                  'compliance_report', 'reasoning_table', 'result')


---

## Step 1 · The input fields of the interface

The model may use more than thirty pieces of information. Putting a field on the screen
for each makes the interface unusable.

The six most important are therefore selected and the rest are held at the typical values
of the training group. The selection is made from the `reasoning_table` you produced in
NB4.

That choice has a cost and it has to appear on screen: the user must know that the
information they cannot see is being held at default values.


### Prompt 1

```
Write code that selects the information to be shown in the interface.

The table named reasoning_table holds the pieces of information sorted by effect. Select
the six most influential numeric ones.

For each, also compute the typical value in the training group and the lowest and highest
reasonable value; these will be the bounds of the sliders in the interface. Do not let
isolated values at the extremes distort the bounds.

Keep the results under these names:
  INTERFACE_FIELDS -> the names of the information to show, as a list
  DEFAULTS         -> the typical value of each
  BOUNDS           -> the lowest and highest value for each

Print which fields were selected and how many remain hidden.

FORMAT
Write a single Python cell. Add a short comment beside each line. Keep it short and
readable; someone who does not know Python should be able to follow it. Show the
steps openly rather than using compact shortcuts. Below the code, summarise what it
does in three plain sentences.

EXPECTED RESULT
There must be three objects named INTERFACE_FIELDS, DEFAULTS and BOUNDS.
INTERFACE_FIELDS must hold six field names.
```


In [ ]:
#@cdss arayuz_alanlari
# Paste the generated code below this line.


### Check 1


In [ ]:
kit.check_defined('INTERFACE_FIELDS', 'DEFAULTS', 'BOUNDS')
print('\nSelected fields:', list(INTERFACE_FIELDS))


---

## Step 2 · The interface

Now you will have the interface itself built.

Note one point. Demographic information such as sex or insurance may appear in the
reasoning list. These are not reasons to present to a clinician and must be moved to a
separate warning line. A model having learned from a demographic attribute is a fairness
audit finding and can prevent deployment.


### Prompt 2

```
Build a single page web interface with Gradio.

I have the following:
  guarded_prediction -> takes one patient's information and returns a dictionary with
                        decision, probability and reason
  patient_reasoning  -> takes one patient's information and returns a table with the
                        columns feature, value and contribution
  INTERFACE_FIELDS   -> the names of the information to show
  DEFAULTS           -> the typical value of each
  BOUNDS             -> the lower and upper bounds of the sliders

The interface should be laid out as follows:
1. A title at the top with this warning: This is a teaching prototype, not a validated
   clinical tool, and it cannot be used for real patient decisions.
2. On the left a slider for each field and an Evaluate button.
3. On the right the decision as text and the reasoning table.
4. State on screen how many fields are not shown and that they are held at default
   values.
5. If the reasoning table holds demographic information such as sex, insurance or
   marital status, remove it from the table and show a separate fairness audit warning
   instead.
6. When the system produces no prediction, that is when a guardrail stops it, state the
   reason plainly.

Write the part that performs the evaluation as a separate piece of work named evaluate,
and have the interface call it. That way we can test it without opening the interface.

On the final line, launch the interface with a shareable link.

FORMAT
Write a single Python cell. Add a short comment beside each line. Keep it short and
readable; someone who does not know Python should be able to follow it. Show the
steps openly rather than using compact shortcuts. Below the code, summarise what it
does in three plain sentences.

EXPECTED RESULT
There must be a runnable piece of work named evaluate that takes the slider values in
order and returns two things: a piece of text and a table.
There must be a Gradio object named interface.
```


In [ ]:
#@cdss arayuz
# Paste the generated code below this line.


### Check 2

The cell below tests the evaluation piece of work directly, without opening the
interface.


In [ ]:
kit.check_function('evaluate')


In [ ]:
inputs = [float(DEFAULTS[f]) for f in INTERFACE_FIELDS]
text, table = evaluate(*inputs)
print('ORDINARY PATIENT'); print(text)
print()
extreme = [float(BOUNDS[f][1]) * 5 for f in INTERFACE_FIELDS]
text_extreme, _ = evaluate(*extreme)
print('PATIENT WITH EXTREME VALUES'); print(text_extreme.split(chr(10))[0])


### Python note · Callbacks

In the interface code you will see a line such as `button.click(evaluate, ...)`. Here
`evaluate` is not being called; it is being **given** to the interface.

This is called a **callback**. The interface runs that piece of work itself when the
button is pressed. Your task is to define the work; the interface decides when it runs.

The distinction is practically useful: because the evaluation was written as a separate
piece, the cell above could test it without opening the interface at all. Keeping the
interface and the working logic apart is an established habit in software development.


---

## Step 3 · Opening the interface


In [ ]:
interface.launch(share=True)


### What to try in the interface

Make five attempts.

Run it at the default values and see what the decision is.

Move the sliders slowly to bring the probability towards the threshold. Find the point at
which the system becomes undecided. In clinical use that band determines who goes on the
list.

Pull several sliders to their extremes. The system should stop producing a prediction.

Reach a similar probability through different slider combinations and watch the reasoning
table change. Two patients may carry the same risk while requiring different action from
the clinician.

Finally, if your interface has one, change a demographic field and touch nothing else. If
the probability moves and a fairness warning appears, the model has learned from that
field.


---

## The complete program

The cell below collects everything you wrote across the six notebooks and saves it as a
single file. That file is your clinical decision support system.

The file disappears when the Colab session closes. Download it from the file panel on the
left.


In [ ]:
program = kit.export(save_as='cdss_system.py')

print()
print('Number of steps:', len(kit.steps()))
print('Lines of code  :', program.count(chr(10)))


## End of the workshop

Across six notebooks you wrote a clinical decision support system. It reaches the data,
prepares it, builds a model, measures its performance, justifies its decisions, applies
safety guardrails, produces a compliance report and runs over the web.

You hand-coded none of it. At each step you described what you wanted, tested the code
that arrived and had it corrected where necessary.

A working prototype does not mean a ready system. What remains: external validation at
another centre, physiological limits supplied by a clinician, a workflow integration
study, approval of the thresholds by the clinical team, regulatory classification, a data
protection assessment and post-deployment performance monitoring.

That list shows where two hours of work sits on the path to a clinical product. The cost
of producing code has fallen; nothing else on the list has.
---

**Warning.** Nothing produced in this notebook is a validated clinical tool. The
MIMIC-IV demo data comes from a single hospital in the United States and does not
represent an intensive care population elsewhere. The material is for teaching.
